Deep Feature extraction using trained custom MobileNetV2 model

In [2]:
import os
print([f for f in os.listdir('.') if f.endswith(('.h5', '.keras'))])

[]


In [3]:
import os
print(os.path.exists('extracted_features_BTTypes'))
print(os.listdir('extracted_features_BTTypes'))

True
['train_features.pkl', 'val_features.pkl', 'test_features.pkl']


In [4]:
print("BTTypes deep features extracted and saved.")

BTTypes deep features extracted and saved.


In [6]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.stats import skew, entropy, kurtosis, variation, iqr
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# Existing BTT deep-feature pickle files
feature_dir = 'extracted_features_BTTypes'
output_root = 'zfmis_selected_features_BTTypes'
os.makedirs(output_root, exist_ok=True)

with open(os.path.join(feature_dir, 'train_features.pkl'), 'rb') as file:
    train_features, train_labels = pickle.load(file)

with open(os.path.join(feature_dir, 'val_features.pkl'), 'rb') as file:
    val_features, val_labels = pickle.load(file)

with open(os.path.join(feature_dir, 'test_features.pkl'), 'rb') as file:
    test_features, test_labels = pickle.load(file)

# Convert one-hot labels to one-dimensional class labels when needed.
def convert_labels_to_1d(labels):
    if len(labels.shape) > 1 and labels.shape[1] > 1:
        return np.argmax(labels, axis=1)
    return labels

# Full 26-feature set from the existing statistical-feature workflow.
def signal_to_noise(feature_vector):
    mean = np.mean(feature_vector)
    std = np.std(feature_vector)
    return mean / (std + 1e-6)


def calculate_statistical_features(features):
    stats_features = []
    for feature_vector in features:
        stats = {
            'mean': np.mean(feature_vector),
            'std_dev': np.std(feature_vector),
            'variance': np.var(feature_vector),
            'median': np.median(feature_vector),
            'range': np.ptp(feature_vector),
            'skewness': skew(feature_vector),
            'kurtosis': kurtosis(feature_vector),
            'entropy': entropy(np.abs(feature_vector) + 1e-6),
            'energy': np.sum(feature_vector ** 2),
            'contrast': np.std(feature_vector) ** 2,
            'mean_abs_dev': np.mean(np.abs(feature_vector - np.mean(feature_vector))),
            'min_value': np.min(feature_vector),
            'max_value': np.max(feature_vector),
            'iqr': iqr(feature_vector),
            'percentile_25': np.percentile(feature_vector, 25),
            'percentile_50': np.percentile(feature_vector, 50),
            'percentile_75': np.percentile(feature_vector, 75),
            'signal_to_noise': signal_to_noise(feature_vector),
            'coef_of_var': variation(feature_vector),
            'autocorrelation': np.corrcoef(feature_vector[:-1], feature_vector[1:])[0, 1]
            if len(feature_vector) > 1 else 0,
            'shannon_entropy': -np.sum(
                feature_vector * np.log2(feature_vector + 1e-6)
            ),
            'root_mean_square': np.sqrt(np.mean(feature_vector ** 2)),
            'harmonic_mean': len(feature_vector) / np.sum(1.0 / (feature_vector + 1e-6)),
            'geometric_mean': np.exp(np.mean(np.log(feature_vector + 1e-6))),
            'std_error_mean': np.std(feature_vector) / np.sqrt(len(feature_vector)),
            'median_abs_dev': np.median(
                np.abs(feature_vector - np.median(feature_vector))
            )
        }
        stats_features.append(stats)
    return pd.DataFrame(stats_features)

# Compute the full statistical tables before ZFMIS filtering.
statistical_tables = {}
for set_name, features, labels in [
    ('train', train_features, train_labels),
    ('val', val_features, val_labels),
    ('test', test_features, test_labels)
]:
    statistical_table = calculate_statistical_features(features)
    statistical_table['label'] = convert_labels_to_1d(labels)
    statistical_tables[set_name] = statistical_table

# Apply zero-feature filtering across splits, while preserving the target column.
def find_zero_features(dataframe):
    return [
        column for column in dataframe.columns
        if column != 'label' and (dataframe[column] == 0).mean() > 0.5
    ]

zero_features = set()
for dataframe in statistical_tables.values():
    zero_features.update(find_zero_features(dataframe))

filtered_tables = {
    set_name: dataframe.drop(columns=zero_features)
    for set_name, dataframe in statistical_tables.items()
}

feature_columns = [
    column for column in filtered_tables['train'].columns
    if column != 'label'
]

# Run SelectKBest(mutual_info_classif) for each requested k.
for k in [3, 6, 9, 12, 15, 18]:
    selector = SelectKBest(score_func=mutual_info_classif, k=k)
    selector.fit(filtered_tables['train'][feature_columns], filtered_tables['train']['label'])
    selected_columns = list(np.array(feature_columns)[selector.get_support()])

    k_output_dir = os.path.join(output_root, f'k_{k}')
    os.makedirs(k_output_dir, exist_ok=True)

    for set_name, dataframe in filtered_tables.items():
        selected_table = dataframe[selected_columns + ['label']]
        output_path = os.path.join(k_output_dir, f'{set_name}_selected_features.csv')
        selected_table.to_csv(output_path, index=False)

    print(f'k={k}: selected {len(selected_columns)} features -> {k_output_dir}')

print(f'ZFMIS completed. Zero-filtered features: {sorted(zero_features)}')
print(f'Outputs saved under: {output_root}')

k=3: selected 3 features -> zfmis_selected_features_BTTypes/k_3
k=6: selected 6 features -> zfmis_selected_features_BTTypes/k_6
k=9: selected 9 features -> zfmis_selected_features_BTTypes/k_9
k=12: selected 12 features -> zfmis_selected_features_BTTypes/k_12
k=15: selected 15 features -> zfmis_selected_features_BTTypes/k_15
k=18: selected 18 features -> zfmis_selected_features_BTTypes/k_18
ZFMIS completed. Zero-filtered features: ['min_value', 'percentile_25']
Outputs saved under: zfmis_selected_features_BTTypes
